# AI Tax + Healthcare Fraud Detection — Exploratory Analysis

This notebook walks through the full dataset from top to bottom:

1. Dataset overview & class balance  
2. Feature distributions by fraud label  
3. Correlation heatmaps  
4. ROC & Precision-Recall curves  
5. Confusion matrix heatmaps  
6. Fraud score distributions  
7. Feature importance (both models)  
8. High-risk record deep-dive  

> **Pre-requisite:** Run all four pipeline steps first (`python src/generate_synthetic_data.py` → `feature_engineering.py` → `train_model.py` → `evaluate_model.py`) or use the Streamlit app.

In [ ]:
# ── 0. Imports & paths ──────────────────────────────────────────────────────
from pathlib import Path

import joblib
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

ROOT     = Path("..").resolve()
RAW_DIR  = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
MDL_DIR  = ROOT / "models"
RPT_DIR  = ROOT / "reports"

TARGET  = "is_fraud"
ID_COLS = {"tax_id", "claim_id", "provider_id", TARGET}

C_LEGIT, C_FRAUD, C_WARN = "#2196F3", "#F44336", "#FF9800"

print("Paths set. ROOT =", ROOT)


## 1. Dataset Overview & Class Balance

In [ ]:

tax_raw  = pd.read_csv(RAW_DIR / "tax_records.csv")
hc_raw   = pd.read_csv(RAW_DIR / "healthcare_claims.csv")

for name, df in [("Tax Records", tax_raw), ("Healthcare Claims", hc_raw)]:
    total  = len(df)
    fraud  = df[TARGET].sum()
    legit  = total - fraud
    print(f"\n{'='*45}")
    print(f"  {name}  ({total:,} records)")
    print(f"{'='*45}")
    print(f"  Legitimate : {legit:,}  ({legit/total:.1%})")
    print(f"  Fraudulent : {fraud:,}  ({fraud/total:.1%})")
    print(df.drop(columns=[TARGET]).describe().round(2).T[
        ["mean","std","min","max"]].to_string())

# Class balance bar charts side by side
fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Tax Records", "Healthcare Claims"])

for col_idx, (df, name) in enumerate(
        [(tax_raw, "Tax"), (hc_raw, "Healthcare")], start=1):
    counts = df[TARGET].value_counts().rename({0:"Legit", 1:"Fraud"})
    fig.add_trace(
        go.Bar(x=counts.index, y=counts.values,
               marker_color=[C_LEGIT, C_FRAUD],
               text=counts.values, textposition="outside",
               name=name, showlegend=False),
        row=1, col=col_idx)

fig.update_layout(title_text="Class Balance — Raw Datasets",
                  height=360, margin=dict(t=60))
fig.show()


## 2. Feature Distributions by Fraud Label

In [ ]:

def distribution_grid(df, features, title, rows=2):
    df2 = df.copy()
    df2["Label"] = df2[TARGET].map({0: "Legit", 1: "Fraud"})
    cols = -(-len(features) // rows)   # ceiling division
    fig = make_subplots(rows=rows, cols=cols,
                        subplot_titles=features)
    for idx, feat in enumerate(features):
        r, c = divmod(idx, cols)
        for label, colour in [("Legit", C_LEGIT), ("Fraud", C_FRAUD)]:
            subset = df2[df2["Label"] == label][feat].dropna()
            fig.add_trace(
                go.Histogram(x=subset, name=label, marker_color=colour,
                             opacity=0.65, nbinsx=40,
                             showlegend=(idx == 0)),
                row=r + 1, col=c + 1)
    fig.update_layout(title_text=title, barmode="overlay",
                      height=260 * rows, margin=dict(t=60))
    fig.show()

# ── Tax features ──────────────────────────────────────────────────────────────
tax_feats = ["reported_income", "total_deductions", "refund_amount",
             "income_gap_ratio", "deduction_to_income_ratio",
             "num_prior_audits", "num_amendments", "occupation_risk_score"]

tax_proc = pd.read_csv(PROC_DIR / "tax_train.csv")
tax_proc["income_gap_ratio"] = (
    (tax_proc["w2_income"] + tax_proc["self_employment_income"] - tax_proc["reported_income"]).clip(0)
    / tax_proc["reported_income"].clip(1)
) if "income_gap_ratio" not in tax_proc.columns else tax_proc["income_gap_ratio"]

distribution_grid(tax_proc,
    ["reported_income","total_deductions","refund_amount",
     "occupation_risk_score","deduction_to_income_ratio",
     "income_gap_ratio","num_prior_audits","num_amendments"],
    "Tax Records — Feature Distributions (Legit vs Fraud)")


In [ ]:

# ── Healthcare features ───────────────────────────────────────────────────────
hc_proc = pd.read_csv(PROC_DIR / "healthcare_train.csv")

distribution_grid(hc_proc,
    ["claim_amount","allowed_amount","upcoding_score",
     "unbundling_score","provider_claim_velocity",
     "amount_to_allowed_ratio","procedure_risk_score","diagnosis_complexity"],
    "Healthcare Claims — Feature Distributions (Legit vs Fraud)")


## 3. Correlation Heatmaps

In [ ]:

def corr_heatmap(df, title):
    num_df = df.select_dtypes(include="number")
    corr   = num_df.corr()
    fig = px.imshow(corr, text_auto=True, aspect="auto",
                    color_continuous_scale="RdBu_r",
                    zmin=-1, zmax=1, title=title)
    fig.update_layout(height=600, margin=dict(t=60))
    fig.show()

corr_heatmap(tax_proc,  "Tax Records — Correlation Matrix (numeric features incl. is_fraud)")
corr_heatmap(hc_proc,   "Healthcare Claims — Correlation Matrix")


## 4. ROC & Precision-Recall Curves

In [ ]:

models_meta = [
    ("tax_fraud_model",        PROC_DIR / "tax_test.csv",         "Tax",         C_WARN),
    ("healthcare_fraud_model", PROC_DIR / "healthcare_test.csv",  "Healthcare",  C_FRAUD),
]

fig_roc = go.Figure()
fig_pr  = go.Figure()

for model_name, test_csv, label, colour in models_meta:
    mdl  = joblib.load(MDL_DIR / f"{model_name}.joblib")
    df   = pd.read_csv(test_csv)
    feat = [c for c in df.columns if c not in ID_COLS]
    y    = df[TARGET]
    yp   = mdl.predict_proba(df[feat])[:, 1]

    fpr, tpr, _ = roc_curve(y, yp)
    auc_roc      = roc_auc_score(y, yp)
    prec, rec, _ = precision_recall_curve(y, yp)
    ap            = average_precision_score(y, yp)

    fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{label} (AUC={auc_roc:.3f})",
                                 line=dict(color=colour, width=2.5)))
    fig_pr.add_trace(go.Scatter(x=rec, y=prec, mode="lines", name=f"{label} (AP={ap:.3f})",
                                line=dict(color=colour, width=2.5)))

fig_roc.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines",
                              line=dict(dash="dash", color="grey"), showlegend=False))
fig_roc.update_layout(title="ROC Curves — both models", xaxis_title="FPR",
                      yaxis_title="TPR", height=420)
fig_roc.show()

fig_pr.update_layout(title="Precision-Recall Curves — both models",
                     xaxis_title="Recall", yaxis_title="Precision", height=420)
fig_pr.show()


## 5. Confusion Matrix Heatmaps

In [ ]:

import json

fig_cms = make_subplots(rows=1, cols=2,
    subplot_titles=["Tax Fraud Model", "Healthcare Fraud Model"])

for col_idx, model_name in enumerate(
        ["tax_fraud_model", "healthcare_fraud_model"], start=1):
    rpt = json.loads((RPT_DIR / f"{model_name}_eval_metrics.json").read_text())
    cm  = rpt["confusion_matrix"]
    z   = [[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]]
    hm  = go.Heatmap(
        z=z, text=[[str(v) for v in row] for row in z],
        texttemplate="%{text}",
        x=["Predicted Legit", "Predicted Fraud"],
        y=["Actual Legit",    "Actual Fraud"],
        colorscale=[[0,"#E3F2FD"],[1,C_FRAUD]],
        showscale=False,
    )
    fig_cms.add_trace(hm, row=1, col=col_idx)

fig_cms.update_layout(title_text="Confusion Matrices — Test Set",
                      height=380, margin=dict(t=60))
fig_cms.show()


## 6. Fraud Score Distributions

In [ ]:

for model_name, test_csv, label, colour in models_meta:
    mdl  = joblib.load(MDL_DIR / f"{model_name}.joblib")
    df   = pd.read_csv(test_csv)
    feat = [c for c in df.columns if c not in ID_COLS]
    df["fraud_score"] = mdl.predict_proba(df[feat])[:, 1]
    df["Label"] = df[TARGET].map({0: "Legit", 1: "Fraud"})

    fig = px.histogram(df, x="fraud_score", color="Label",
                       color_discrete_map={"Legit": C_LEGIT, "Fraud": C_FRAUD},
                       nbins=60, barmode="overlay", opacity=0.70,
                       title=f"{label} — Fraud Score Distribution (Test Set)",
                       labels={"fraud_score": "Fraud Probability"})
    fig.add_vline(x=0.5, line_dash="dash", line_color="black",
                  annotation_text="Decision threshold = 0.5")
    fig.update_layout(height=360, margin=dict(t=60))
    fig.show()


## 7. Feature Importance

In [ ]:

TOP_N = 15

for model_name, test_csv, label, colour in models_meta:
    mdl  = joblib.load(MDL_DIR / f"{model_name}.joblib")
    df   = pd.read_csv(test_csv)
    feat_cols = [c for c in df.columns if c not in ID_COLS]
    rf   = mdl.named_steps["model"]
    imp  = (pd.Series(rf.feature_importances_, index=feat_cols)
              .nlargest(TOP_N)
              .sort_values())

    fig = px.bar(imp, x=imp.values, y=imp.index, orientation="h",
                 color=imp.values,
                 color_continuous_scale=[[0,"#E3F2FD"],[1, colour]],
                 title=f"{label} Model — Top {TOP_N} Feature Importances",
                 labels={"x": "Mean Decrease in Impurity", "y": ""})
    fig.update_layout(height=460, margin=dict(t=60), coloraxis_showscale=False)
    fig.show()


## 8. High-Risk Record Deep-Dive

Inspect the top-20 highest-scoring predicted frauds from each test set and compare their feature values against the population mean.

In [ ]:

for model_name, test_csv, label, colour in models_meta:
    mdl  = joblib.load(MDL_DIR / f"{model_name}.joblib")
    df   = pd.read_csv(test_csv)
    feat_cols = [c for c in df.columns if c not in ID_COLS]

    df["fraud_score"] = mdl.predict_proba(df[feat_cols])[:, 1]
    top20 = df.nlargest(20, "fraud_score")[feat_cols + ["fraud_score", TARGET]]
    pop_mean = df[feat_cols].mean()

    # Z-score of top20 vs population mean (replace avoids div-by-zero on zero-variance cols)
    _std: pd.Series = df[feat_cols].std()  # type: ignore[assignment]
    pop_std = _std.clip(lower=1)
    z = ((top20[feat_cols].mean() - pop_mean) / pop_std).sort_values(ascending=False)

    fig = px.bar(z, x=z.values, y=z.index, orientation="h",
                 color=z.values,
                 color_continuous_scale=[[0, "#E3F2FD"], [0.5, "#FFF9C4"], [1, C_FRAUD]],
                 title=f"{label} — Top-20 High-Risk Records: Feature Z-Scores vs Population",
                 labels={"x": "Z-score (vs population mean)", "y": ""})
    fig.add_vline(x=0, line_dash="solid", line_color="black", line_width=1)
    fig.update_layout(height=500, margin=dict(t=60), coloraxis_showscale=False)
    fig.show()

    print(f"\n{label} — Top 5 highest-score records:")
    display(top20.head())


## 9. Summary Table — Both Models

Side-by-side evaluation metrics for easy comparison.

In [ ]:

rows = []
for model_name, _, label, _ in models_meta:
    rpt = json.loads((RPT_DIR / f"{model_name}_eval_metrics.json").read_text())
    cm  = rpt.pop("confusion_matrix", {})
    rpt["domain"] = label
    rpt["tn"], rpt["fp"] = cm.get("tn"), cm.get("fp")
    rpt["fn"], rpt["tp"] = cm.get("fn"), cm.get("tp")
    rows.append(rpt)

summary = pd.DataFrame(rows).set_index("domain")
display(summary.T.style
    .format("{:.4f}", subset=pd.IndexSlice[
        ["roc_auc","pr_auc","precision","recall","f1","fraud_rate"], :])
    .background_gradient(cmap="RdYlGn", axis=1,
        subset=pd.IndexSlice[["roc_auc","pr_auc","f1"], :])
    .set_caption("Model Evaluation Summary — Test Set"))
